In [1]:
!spark-submit --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.0
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 17.0.8.1
Branch HEAD
Compiled by user ubuntu on 2023-09-09T01:53:20Z
Revision ce5ddad990373636e94071e7cef2f31021add07b
Url https://github.com/apache/spark
Type --help for more information.


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Estimator, Model
from pyspark.ml.param.shared import HasMaxIter, HasLearningRate
from pyspark.ml.param import Param, Params, TypeConverters
from pyspark.ml.regression import LinearRegression

import numpy as np
import pandas as pd
import unittest
import os

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .getOrCreate()

In [4]:
rng = np.random.default_rng(42)

num_samples = 2_500_000
num_features = 10

w_tr = rng.standard_normal(num_features)
X = rng.standard_normal((num_samples, num_features))
noise = rng.standard_normal(num_samples) * 0.05
y = np.dot(X, w_tr) + noise

In [5]:
X.nbytes + y.nbytes

220000000

In [6]:
data = np.column_stack((X, y))
feature_cols = [f"feature{i+1}" for i in range(num_features)]
columns = feature_cols + ["label"]

In [7]:
df_pd = pd.DataFrame(data, columns=columns)
df_pd.to_parquet("data.parquet", index=False)

In [8]:
file_size = os.path.getsize("data.parquet")
print(f"{file_size / 1_000_000:.2f} МБ")

229.24 МБ


In [9]:
df = spark.read.parquet("data.parquet")

In [10]:
vector = VectorAssembler(inputCols=feature_cols, outputCol='features')
df = vector.transform(df)

In [11]:
df_model = df.select("features", "label")

### Реализация Linear Regression

In [12]:
class MyParams(HasLearningRate, HasMaxIter):
    def __init__(self):
        super(MyParams, self).__init__()
        HasLearningRate.__init__(self)
        HasMaxIter.__init__(self)

    def setParams(self, learningRate, maxIter):
        self._set(learningRate=learningRate, maxIter=maxIter)
        return self

In [13]:
class MyModel(Model, MyParams):
    def __init__(self, weights, bias):
        super(MyModel, self).__init__()
        MyParams.__init__(self)
        self.coefficients = weights
        self.intercept = bias

    def _transform(self, dataset):
        def predict(row):
            features = row.features
            prediction = float(np.dot(self.coefficients, features) + self.intercept)
            return (float(row.label), prediction)

        rdd = dataset.rdd.map(predict)
        return dataset.sparkSession.createDataFrame(rdd, ["label", "prediction"])

In [14]:
class MyEstimator(Estimator, MyParams):
    def __init__(self):
        super(MyEstimator, self).__init__()
        MyParams.__init__(self)

    def _fit(self, dataset):
        lr = self.getLearningRate()
        maxIter = self.getMaxIter()

        rdd = dataset.rdd.map(lambda row: (np.array(row.features), row.label)).cache()
        num_features = len(rdd.first()[0])
        n = rdd.count()

        weights = np.zeros(num_features)
        bias = 0.0

        for epoch in range(maxIter):
            def compute_gradients(row):
                x, y = row
                y_pred = np.dot(weights, x) + bias
                error = y_pred - y
                return x * error, error

            grad_sum = rdd.map(compute_gradients).reduce(
                lambda a, b: (a[0] + b[0], a[1] + b[1])
            )

            weights -= lr * grad_sum[0] / n
            bias -= lr * grad_sum[1] / n

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}: coefficients={weights}, intercept={bias}")

        model = MyModel(weights, bias)
        model.setParams(learningRate=lr, maxIter=maxIter)
        rdd.unpersist()
        return model

In [15]:
estimator = MyEstimator().setParams(learningRate=0.1, maxIter=100)
my_model = estimator.fit(df_model)

Epoch 10: coefficients=[ 0.1980408  -0.67787054  0.48851612  0.61217227 -1.26905397 -0.84854344
  0.08298169 -0.20588797 -0.01037768 -0.55575282], intercept=-0.0003820380645557741
Epoch 20: coefficients=[ 0.26735244 -0.913903    0.65901963  0.82592262 -1.71261915 -1.14414299
  0.11210746 -0.27775749 -0.01437147 -0.74944862], intercept=-0.0002714338315086865
Epoch 30: coefficients=[ 0.2916104  -0.99608874  0.71852943  0.90055702 -1.86765611 -1.24711866
  0.12233032 -0.30284519 -0.01589523 -0.81695688], intercept=-0.00014742542411950164
Epoch 40: coefficients=[ 0.30010024 -1.02470554  0.73929982  0.92661677 -1.92184541 -1.28299157
  0.12591843 -0.31160267 -0.0164724  -0.84048524], intercept=-7.424381882280656e-05
Epoch 50: coefficients=[ 0.3030715  -1.03466981  0.7465492   0.93571591 -1.94078595 -1.2954884
  0.12717782 -0.31465969 -0.01668969 -0.84868545], intercept=-3.823416412928743e-05
Epoch 60: coefficients=[ 0.30411138 -1.03813933  0.74907943  0.93889301 -1.94740616 -1.29984185
  0.

In [16]:
lr = LinearRegression(featuresCol="features", labelCol="label")
model = lr.fit(df_model)

### Тесты

In [20]:
class TestModelComparison(unittest.TestCase):

    def test_weights_and_intercept(self):
        np.testing.assert_allclose(
            model.coefficients.toArray(),
            my_model.coefficients,
            rtol=1e-4,
            err_msg="Coefficients моделей не совпадают"
        )
        self.assertAlmostEqual(
            model.intercept,
            my_model.intercept,
            places=4,
            msg="Intercept моделей не совпадают"
        )

    def test_weights_vs_true_weights(self):
        np.testing.assert_allclose(
            my_model.coefficients,
            w_tr,
            rtol=1e-2,
            err_msg="Coefficients модели не совпадают с w_tr"
        )

    def test_predictions(self):
        lr_preds = model.transform(df_model).select("prediction").rdd.map(lambda row: round(row.prediction, 4)).collect()
        my_preds = my_model.transform(df_model).select("prediction").rdd.map(lambda row: round(row.prediction, 4)).collect()

        for i, (p_lr, p_my) in enumerate(zip(lr_preds, my_preds)):
            self.assertAlmostEqual(p_lr, p_my, places=3, msg=f"Не совпадают {i}")

suite = unittest.TestLoader().loadTestsFromTestCase(TestModelComparison)
unittest.TextTestRunner(verbosity=2).run(suite)

test_predictions (__main__.TestModelComparison.test_predictions) ... ok
test_weights_and_intercept (__main__.TestModelComparison.test_weights_and_intercept) ... ok
test_weights_vs_true_weights (__main__.TestModelComparison.test_weights_vs_true_weights) ... ok

----------------------------------------------------------------------
Ran 3 tests in 29.158s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>